# 02 Text Chunking

Purpose: convert cleaned papers into chunk-level records for retrieval.

Input: `1_data/processed/papers_clean.parquet`
Output: `1_data/processed/chunks.parquet`, `1_data/processed/chunk_config.json`

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('..').resolve()
IN_PATH = ROOT / '1_data' / 'processed' / 'papers_clean.parquet'
OUT_DIR = ROOT / '1_data' / 'processed'

CHUNK_SIZE = 400
OVERLAP = 50
STEP = max(CHUNK_SIZE - OVERLAP, 1)

assert IN_PATH.exists(), f'Missing file: {IN_PATH}'
df = pd.read_parquet(IN_PATH)
records = []

for _, row in df.iterrows():
    text = f"{row.get('title', '')} {row.get('abstract', '')}".strip()
    words = text.split()
    if not words:
        continue
    for start in range(0, len(words), STEP):
        chunk_words = words[start:start + CHUNK_SIZE]
        if len(chunk_words) < 20:
            continue
        idx = start // STEP
        records.append({
            'chunk_id': f"{row.get('cord_uid', 'unknown')}_chunk_{idx:03d}",
            'cord_uid': row.get('cord_uid', ''),
            'title': row.get('title', ''),
            'chunk_text': ' '.join(chunk_words),
            'chunk_index': idx
        })

chunks = pd.DataFrame(records)
out_path = OUT_DIR / 'chunks.parquet'
chunks.to_parquet(out_path, index=False)
(OUT_DIR / 'chunk_config.json').write_text(json.dumps({'chunk_size': CHUNK_SIZE, 'overlap': OVERLAP}, indent=2))
print(f'Wrote {len(chunks):,} chunks -> {out_path}')
